# AHA-Fed on NIH ChestX-ray14 (full resolution)

The paper is motivated by chest radiography but its real-data results are on
28×28 MedMNIST. This notebook runs the federation on the actual NIH
ChestX-ray14 corpus — 112,120 images, 14 findings, patient-disjoint centers.

**Settings:** Accelerator → GPU T4 x2, Internet → On.

**Inputs (both required):**
- `aha-fed-code` — the repo
- `nih-chest-xrays/data` — the 45 GB NIH corpus, already hosted on Kaggle
  (Add Input → search "NIH Chest X-rays" → the `nih-chest-xrays/data` one).
  Nothing is downloaded; Kaggle mounts it read-only.

**Runtime is the real constraint.** 112k images at 224×224 through DenseNet-121
will not finish a multi-seed sweep inside one 9 h session. The plan below runs
one seed per session and resumes, rather than pretending it all fits.

In [ ]:
import glob, os, shutil, sys, zipfile

REPO = '/kaggle/working/repo'

def find_repo():
    for hit in glob.glob('/kaggle/input/**/paper/build_paper.py', recursive=True):
        return os.path.dirname(os.path.dirname(hit)), 'tree'
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if zips:
        return zips[0], 'zip'
    raise SystemExit('Attach the aha-fed-code dataset.')

if not os.path.isdir(os.path.join(REPO, 'paper')):
    src, kind = find_repo()
    os.makedirs(REPO, exist_ok=True)
    if kind == 'zip':
        with zipfile.ZipFile(src) as z:
            z.extractall(REPO)
    else:
        shutil.copytree(src, REPO, dirs_exist_ok=True)

os.chdir(REPO)
sys.path.insert(0, REPO)
print('cwd:', os.getcwd())

In [ ]:
# Kaggle's torch 2.10 is built for sm_70+ and cannot run on the P100 (sm_60)
# that sessions are sometimes assigned. Probe with a subprocess so the check
# does not pin a torch version into this kernel, then install a compatible
# build before anything imports torch here.
import subprocess, sys

PROBE = (
    'import torch;'
    'a=torch.randn(4,3,8,8,device="cuda");'
    'w=torch.randn(2,3,3,3,device="cuda");'
    'torch.nn.functional.conv2d(a,w).sum().item();'
    'print(torch.cuda.get_device_name(0), torch.__version__)'
)
r = subprocess.run([sys.executable, '-c', PROBE],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])

if r.returncode != 0:
    print('GPU cannot run the installed torch; installing cu118 build ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch==2.4.1', 'torchvision==0.19.1',
                    '--index-url', 'https://download.pytorch.org/whl/cu118'],
                   check=True)
    r2 = subprocess.run([sys.executable, '-c', PROBE],
                        capture_output=True, text=True)
    print(r2.stdout.strip() or r2.stderr.strip()[-400:])
    if r2.returncode != 0:
        raise SystemExit('no working torch/GPU combination on this session')
    print('cu118 build works on this GPU')
else:
    print('default torch works on this GPU')


## Locate the NIH corpus

Kaggle stores the images in twelve `images_0NN/images/` folders. Our loader
wants one flat directory, so we build a symlink farm instead of copying 45 GB.

In [ ]:
import glob, os

csvs = glob.glob('/kaggle/input/**/Data_Entry_2017*.csv', recursive=True)
assert csvs, 'NIH dataset not attached — Add Input -> nih-chest-xrays/data'
CSV = csvs[0]
NIH_ROOT = os.path.dirname(CSV)
print('csv :', CSV)

shards = sorted(glob.glob(f'{NIH_ROOT}/images_*/images'))
print(f'found {len(shards)} image shards')

FLAT = '/kaggle/working/nih_images'
os.makedirs(FLAT, exist_ok=True)
if len(os.listdir(FLAT)) < 100000:
    n = 0
    for sh in shards:
        for name in os.listdir(sh):
            dst = os.path.join(FLAT, name)
            if not os.path.exists(dst):
                os.symlink(os.path.join(sh, name), dst)
                n += 1
    print(f'linked {n} images')
print('total in flat dir:', len(os.listdir(FLAT)))

## Sanity check the patient-disjoint federation

In [ ]:
from src.data.nih_chestxray import federate_nih_by_patient

centers = federate_nih_by_patient(csv_path=CSV, image_dir=FLAT, num_centers=4)
print('center sizes:', [len(c) for c in centers])

x, y = centers[0][0]
print('image tensor:', tuple(x.shape), '| label vector:', tuple(y.shape))
print('positives in this label vector:', int(y.sum()))

## Time one round before committing

Measure first. If a round takes T minutes, a 25-round run takes 25·T, and that
decides how many seeds fit in a session — do not guess.

In [ ]:
# The timing probe was replaced, but it also defined the imports and
# constants the run cell needs, so they are set up here instead.
import torch
from torch.utils.data import DataLoader, Subset
from src.federated.client import FederatedClient
from src.federated.server import FederatedServer, build_aggregator
from src.models.backbones import build_model
from src.utils.seed import set_seed

NUM_CENTERS, NUM_CLASSES = 4, 14
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| centers:', len(centers))
print('timing probe skipped: measured 14.1 min/round at 4000 imgs/center')


## The actual run

Set `SEED` and `IMAGES_PER_CENTER` from what the probe reported, then run. One
seed per session; change `SEED` and re-run the notebook to add another. Results
are keyed by seed so they accumulate rather than overwrite.

In [ ]:
import json
from pathlib import Path
from torch.utils.data import Subset

SEED = 42
IMAGES_PER_CENTER = 2000    # 14.1 min/round at 4000 -> ~7 min at 2000
ROUNDS = 15

OUT = Path('/kaggle/working/runs/paper/nih')
OUT.mkdir(parents=True, exist_ok=True)

AGGS = {
    'fedavg':  {'aggregator': 'fedavg'},
    'fedbn':   {'aggregator': 'fedbn'},
    'aha_fed': {'aggregator': 'adaptive', 'alpha': 1.0, 'beta': 4.0, 'gamma': 4.0},
}

eval_loader = DataLoader(Subset(centers[0], range(2000)), batch_size=64)

def save(tag, hist):
    """Write after every evaluation, so a session timeout still leaves
    a usable partial history rather than nothing."""
    (OUT / f'nih_{tag}_s{SEED}.json').write_text(json.dumps({
        'config': {'aggregator': tag, 'seed': SEED,
                   'dataset': 'nih_chestxray14', 'num_centers': NUM_CENTERS,
                   'images_per_center': IMAGES_PER_CENTER,
                   'num_rounds': ROUNDS, 'rounds_completed': len(hist),
                   'resolution': 224, 'backbone': 'densenet121'},
        'history': hist, 'final': hist[-1] if hist else None,
    }, indent=2, default=str))

for tag, kw in AGGS.items():
    out = OUT / f'nih_{tag}_s{SEED}.json'
    if out.exists() and json.loads(out.read_text())['config'].get(
            'rounds_completed', 0) >= ROUNDS:
        print(f'[skip] {out.name}')
        continue
    set_seed(SEED)
    shards = [Subset(c, range(min(IMAGES_PER_CENTER, len(c)))) for c in centers]
    cl = [FederatedClient(i, ds, batch_size=32, device=device, multilabel=True)
          for i, ds in enumerate(shards)]
    agg = (build_aggregator('adaptive', alpha=1.0, beta=4.0, gamma=4.0)
           if tag == 'aha_fed' else build_aggregator(kw['aggregator']))
    srv = FederatedServer(
        global_model=build_model('densenet121', num_classes=NUM_CLASSES,
                                 pretrained=True),
        clients=cl, aggregator=agg, eval_loader=eval_loader,
        num_classes=NUM_CLASSES, device=device, multilabel=True)
    hist = []
    srv.run(num_rounds=ROUNDS, local_epochs=1, local_lr=0.01, eval_every=1,
            round_callback=lambda m: (hist.append(m), save(tag, hist)))
    print(f'[done] {out.name} ({len(hist)} rounds)')


In [ ]:
import glob, json
print(f"{'run':<26}{'AUROC':<10}{'ECE':<10}accuracy")
for f in sorted(glob.glob('/kaggle/working/runs/paper/nih/*.json')):
    d = json.load(open(f))
    fin = d.get('final') or {}
    au = fin.get('auroc')
    print(f"{f.split('/')[-1]:<26}{au if au is None else round(au,4):<10}"
          f"{round(fin.get('ece', 0), 4):<10}{round(fin.get('accuracy', 0), 4)}")


In [ ]:
!cd /kaggle/working && zip -qr nih_results.zip runs/paper/nih
!ls -lh /kaggle/working/nih_results.zip
